# Self-history autoregressive ablation

This mirrors TEP notebook 4.2. Each endogenous process variable is predicted using only its own lag-2 history, with residuals normalized by their healthy-training variances. The moving-average window and consecutive-alarm count are fixed to six and five; the threshold is the maximum healthy-phase validation score across faulty runs.

In [1]:
from pathlib import Path
import sys


def find_project_root(start=Path.cwd()):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / "pyproject.toml").is_file() and (candidate / "data").is_dir():
            return candidate
    raise FileNotFoundError("Could not locate the project root from the notebook working directory.")


PROJECT_ROOT = find_project_root()
sys.path.insert(0, str(PROJECT_ROOT))
from IPython.display import display
from carl.causalai_baselines import run_ablation_comparison

ablation_results = run_ablation_comparison("self_history", PROJECT_ROOT)
display(ablation_results["selected_settings"])

,Ablation,Score,Max Lag,Window Size,Consecutive Alarms,Threshold
0,self_history,diagonal,2,6,5,3.495081


# 2. Threshold Selection (Ablation Settings)

The alarm parameters are held at six moving-average samples and five consecutive alarms. The score threshold is the maximum pre-fault moving average on the faulty validation trajectories.

In [2]:
display(ablation_results["validation_metrics"])

,Ablation,Score,Max Lag,Window Size,Consecutive Alarms,Threshold,Sample Accuracy,Sample TPR,Sample FPR,Sample F1,Trajectory-Phase Accuracy,Trajectory-Phase TPR,Trajectory-Phase FPR,Trajectory-Phase F1,Detected Runs,Total Faulty Runs,Median Anomaly Detection Delay
0,self_history,diagonal,2,6,5,3.495081,0.411626,0.272622,0.0,0.428442,0.795556,0.591111,0.0,0.743017,133,225,34.0


# 3. Test on Faulty Time Series (Ablation Settings)

Evaluate self-history detection on every faulty held-out test run.

In [3]:
display(ablation_results["faulty_test_metrics"])
display(ablation_results["faulty_test_by_type"])

,Ablation,Score,Max Lag,Window Size,Consecutive Alarms,Threshold,Sample Accuracy,Sample TPR,Sample FPR,Sample F1,Trajectory-Phase Accuracy,Trajectory-Phase TPR,Trajectory-Phase FPR,Trajectory-Phase F1,Detected Runs,Total Faulty Runs,Median Anomaly Detection Delay
0,self_history,diagonal,2,6,5,3.495081,0.443433,0.311944,0.0,0.475545,0.782222,0.564444,0.0,0.721591,127,225,32.0


,Fault Type,Runs,Detected_Runs,Run_Detection_Rate,Median_Detection_Delay
0,hydraulic_coupling_shift,75,3,0.040000,559.0
1,reaction_coupling_shift,75,75,1.000000,21.0
2,thermal_coupling_shift,75,49,0.653333,55.0


# 4. Test on Fault-Free Time Series (Ablation Settings)

Evaluate the same settings on all healthy held-out test runs.

In [4]:
display(ablation_results["healthy_test_metrics"])
display(ablation_results["comparison"])

,Ablation,Score,Max Lag,Window Size,Consecutive Alarms,Threshold,Sample Accuracy,Sample TPR,Sample FPR,Sample F1,Trajectory-Phase FPR,False Alarm Runs,Total Healthy Runs
0,self_history,diagonal,2,6,5,3.495081,1.0,0.0,0.0,0.0,0.0,0,75


,Ablation,Score,Max Lag,Window Size,Consecutive Alarms,Threshold,Faulty Test Sample F1,Faulty Test Trajectory-Phase F1,Faulty Test Trajectory-Phase TPR,Faulty Test Median Detection Delay,Healthy Test Sample FPR,Healthy Test Trajectory-Phase FPR
0,full_var,mahalanobis,2,6,5,3.429900,0.566890,0.891626,0.804444,31.0,0.0,0.0
1,pca,spe,2,6,5,2.360392,0.362259,0.821990,0.697778,52.0,0.0,0.0
2,self_history,diagonal,2,6,5,3.495081,0.475545,0.721591,0.564444,32.0,0.0,0.0
